# Hyperspectral Image Classification with CNN and Transformer

**Author:** Alessandro Carotenuto  
**Student ID:** 1847282  
**Dataset:** Pavia University  

## Project aim

The project aims to classify land-cover pixels in a hyperspectral image.
A spatial-spectral patch centered on each labeled pixel is used as input.
A CNN learns local features, while a lightweight Transformer models non-local context.

## Setup 

### Paths

In [1]:
from pathlib import Path
import numpy as np

project_root = Path.cwd().parent
data_dir = project_root / "data" / "raw"

cube_path = data_dir / "PaviaU.mat"
ground_truth_path = data_dir / "PaviaU_gt.mat"

print("Cube:", cube_path)
print("Cube exists:", cube_path.exists())

print("Ground truth:", ground_truth_path)
print("Ground truth exists:", ground_truth_path.exists())

Cube: c:\Users\alex1\Documents\Cloned Repositories\Hyperspectral-Neural-Networks-Project\data\raw\PaviaU.mat
Cube exists: True
Ground truth: c:\Users\alex1\Documents\Cloned Repositories\Hyperspectral-Neural-Networks-Project\data\raw\PaviaU_gt.mat
Ground truth exists: True


### Loading MATLAB Files

In [2]:
from scipy.io import loadmat

cube_file = loadmat(cube_path)
ground_truth_file = loadmat(ground_truth_path)

print(cube_file.keys())
print(ground_truth_file.keys())

dict_keys(['__header__', '__version__', '__globals__', 'paviaU'])
dict_keys(['__header__', '__version__', '__globals__', 'paviaU_gt'])


### Dimension Check

In [3]:
cube = cube_file["paviaU"]
ground_truth = ground_truth_file["paviaU_gt"]

print("Cube:", cube.shape, cube.dtype)
print("Ground truth:", ground_truth.shape, ground_truth.dtype)
print("Cube range:", cube.min(), cube.max())
print("Labels:", np.unique(ground_truth))

Cube: (610, 340, 103) uint16
Ground truth: (610, 340) uint8
Cube range: 0 8000
Labels: [0 1 2 3 4 5 6 7 8 9]


### Class Count

In [4]:
class_names = {
    0: "Unlabeled",
    1: "Asphalt",
    2: "Meadows",
    3: "Gravel",
    4: "Trees",
    5: "Painted metal sheets",
    6: "Bare Soil",
    7: "Bitumen",
    8: "Self-Blocking Bricks",
    9: "Shadows",
}

labels, counts = np.unique(ground_truth, return_counts=True)

for label, count in zip(labels, counts):
    print(label, class_names[label], count)

0 Unlabeled 164624
1 Asphalt 6631
2 Meadows 18649
3 Gravel 2099
4 Trees 3064
5 Painted metal sheets 1345
6 Bare Soil 5029
7 Bitumen 1330
8 Self-Blocking Bricks 3682
9 Shadows 947


### Patch Configuration

In [5]:
from enum import IntEnum


class PatchSize(IntEnum):
    SMALLEST = 7
    SMALL = 11
    PAPER = 15
    LARGE = 19
    LARGEST = 23


PATCH_SIZE = PatchSize.PAPER
PATCH_RADIUS = int(PATCH_SIZE) // 2

print("Patch size:", int(PATCH_SIZE))
print("Patch radius:", PATCH_RADIUS)

Patch size: 15
Patch radius: 7


## Dataset Exploration

### Spectral Bands and Ground Truth

In [ ]:
import matplotlib.pyplot as plt


bands_to_show = [10, 50, 90]

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

for axis, band_index in zip(axes[:3], bands_to_show):
    axis.imshow(cube[:, :, band_index], cmap="gray")
    axis.set_title(f"Band {band_index}")
    axis.axis("off")

ground_truth_image = axes[3].imshow(
    ground_truth,
    cmap="tab10",
    vmin=0,
    vmax=9,
)

axes[3].set_title("Ground Truth")
axes[3].axis("off")

plt.colorbar(ground_truth_image, ax=axes[3], fraction=0.046)
plt.tight_layout()
plt.show()

### Labeled Class Distribution

In [ ]:
labeled_mask = labels != 0
labeled_labels = labels[labeled_mask]
labeled_counts = counts[labeled_mask]
labeled_names = [class_names[label] for label in labeled_labels]

fig, axis = plt.subplots(figsize=(10, 6))
bars = axis.barh(labeled_names, labeled_counts, color="steelblue")

axis.set_title("Pavia University Labeled Class Distribution")
axis.set_xlabel("Number of labeled pixels")
axis.set_ylabel("Class")
axis.invert_yaxis()
axis.bar_label(bars, padding=4)

plt.tight_layout()
plt.show()

### Mean Spectral Signatures

In [ ]:
num_bands = cube.shape[2]
band_indices = np.arange(1, num_bands + 1)

fig, axis = plt.subplots(figsize=(12, 7))

for label in range(1, 10):
    class_pixels = cube[ground_truth == label]
    mean_signature = class_pixels.mean(axis=0)

    axis.plot(
        band_indices,
        mean_signature,
        label=class_names[label],
    )

axis.set_title("Mean Spectral Signature per Class")
axis.set_xlabel("Spectral band")
axis.set_ylabel("Mean pixel value")
axis.grid(alpha=0.25)
axis.legend(bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()

### Coordinates By Class

In [22]:
coordinates_by_class = {
    label: np.argwhere(ground_truth == label)
    for label in range(1, 10)
}

for label, coordinates in coordinates_by_class.items():
    expected_count = counts[labels == label].item()

    assert coordinates.shape == (expected_count, 2)
    print(
        f"{label}: {class_names[label]} - "
        f"{len(coordinates)} coordinates"
    )

total_coordinates = sum(
    len(coordinates)
    for coordinates in coordinates_by_class.values()
)

assert total_coordinates == np.count_nonzero(ground_truth)
print("Total labeled coordinates:", total_coordinates)

1: Asphalt - 6631 coordinates
2: Meadows - 18649 coordinates
3: Gravel - 2099 coordinates
4: Trees - 3064 coordinates
5: Painted metal sheets - 1345 coordinates
6: Bare Soil - 5029 coordinates
7: Bitumen - 1330 coordinates
8: Self-Blocking Bricks - 3682 coordinates
9: Shadows - 947 coordinates
Total labeled coordinates: 42776


### Set Seed

In [23]:
SPLIT_SEED = 42
rng = np.random.default_rng(SPLIT_SEED)

### Few-Shot Coordinate Split

In [ ]:
TRAIN_SAMPLES_PER_CLASS = 10
VALIDATION_SAMPLES_PER_CLASS = 5

# Recreate the generator so rerunning this cell produces the same split.
rng = np.random.default_rng(SPLIT_SEED)

train_coordinates_by_class = {}
validation_coordinates_by_class = {}
test_coordinates_by_class = {}

for label, coordinates in coordinates_by_class.items():
    shuffled_coordinates = rng.permutation(coordinates)
    validation_end = (
        TRAIN_SAMPLES_PER_CLASS + VALIDATION_SAMPLES_PER_CLASS
    )

    train_coordinates_by_class[label] = shuffled_coordinates[
        :TRAIN_SAMPLES_PER_CLASS
    ]
    validation_coordinates_by_class[label] = shuffled_coordinates[
        TRAIN_SAMPLES_PER_CLASS:validation_end
    ]
    test_coordinates_by_class[label] = shuffled_coordinates[
        validation_end:
    ]

    print(
        f"{label}: {class_names[label]} - "
        f"train {len(train_coordinates_by_class[label])}, "
        f"validation {len(validation_coordinates_by_class[label])}, "
        f"test {len(test_coordinates_by_class[label])}"
    )

### Split Verification

In [ ]:
def coordinate_set(coordinates):
    return {tuple(coordinate) for coordinate in coordinates.tolist()}


all_train_coordinates = set()
all_validation_coordinates = set()
all_test_coordinates = set()

for label in range(1, 10):
    original_set = coordinate_set(coordinates_by_class[label])
    train_set = coordinate_set(train_coordinates_by_class[label])
    validation_set = coordinate_set(
        validation_coordinates_by_class[label]
    )
    test_set = coordinate_set(test_coordinates_by_class[label])

    assert len(train_set) == TRAIN_SAMPLES_PER_CLASS
    assert len(validation_set) == VALIDATION_SAMPLES_PER_CLASS
    assert train_set.isdisjoint(validation_set)
    assert train_set.isdisjoint(test_set)
    assert validation_set.isdisjoint(test_set)
    assert train_set | validation_set | test_set == original_set

    all_train_coordinates.update(train_set)
    all_validation_coordinates.update(validation_set)
    all_test_coordinates.update(test_set)

assert all_train_coordinates.isdisjoint(all_validation_coordinates)
assert all_train_coordinates.isdisjoint(all_test_coordinates)
assert all_validation_coordinates.isdisjoint(all_test_coordinates)
assert len(all_train_coordinates) == 90
assert len(all_validation_coordinates) == 45
assert len(all_test_coordinates) == total_coordinates - 135

print("Train coordinates:", len(all_train_coordinates))
print("Validation coordinates:", len(all_validation_coordinates))
print("Test coordinates:", len(all_test_coordinates))
print("All split checks passed.")

### Save the Few-Shot Split

In [ ]:
def build_split_table(coordinates_by_label):
    rows = []

    for label in range(1, 10):
        coordinates = coordinates_by_label[label]
        label_column = np.full((len(coordinates), 1), label)
        rows.append(np.column_stack((coordinates, label_column)))

    return np.vstack(rows)


split_tables = {
    "train": build_split_table(train_coordinates_by_class),
    "validation": build_split_table(
        validation_coordinates_by_class
    ),
    "test": build_split_table(test_coordinates_by_class),
}

split_directory = project_root / "data" / "splits" / f"seed_{SPLIT_SEED}"
split_directory.mkdir(parents=True, exist_ok=True)

for split_name, split_table in split_tables.items():
    output_path = split_directory / f"{split_name}.csv"
    np.savetxt(
        output_path,
        split_table,
        fmt="%d",
        delimiter=",",
        header="row,column,label",
        comments="",
    )
    print(f"Saved {len(split_table)} samples to {output_path}")

### Reload and Verify the Saved Split

In [ ]:
loaded_split_tables = {}

for split_name, expected_table in split_tables.items():
    input_path = split_directory / f"{split_name}.csv"
    loaded_table = np.loadtxt(
        input_path,
        delimiter=",",
        skiprows=1,
        dtype=np.int64,
    )
    loaded_split_tables[split_name] = loaded_table

    assert loaded_table.shape == expected_table.shape
    assert np.array_equal(loaded_table, expected_table)

    rows = loaded_table[:, 0]
    columns = loaded_table[:, 1]
    saved_labels = loaded_table[:, 2]
    assert np.array_equal(ground_truth[rows, columns], saved_labels)

    print(f"Verified {split_name}.csv: {len(loaded_table)} samples")

print(f"Saved split verified successfully for seed {SPLIT_SEED}.")

### Border Analysis for Patch Extraction

In [ ]:
image_height, image_width = ground_truth.shape

for split_name, split_table in loaded_split_tables.items():
    rows = split_table[:, 0]
    columns = split_table[:, 1]

    requires_padding = (
        (rows < PATCH_RADIUS)
        | (rows >= image_height - PATCH_RADIUS)
        | (columns < PATCH_RADIUS)
        | (columns >= image_width - PATCH_RADIUS)
    )

    border_count = np.count_nonzero(requires_padding)
    print(
        f"{split_name}: {border_count} of {len(split_table)} samples "
        "require padding"
    )

### Per-Band Min-Max Normalization

In [ ]:
cube_float = cube.astype(np.float32)

band_min = cube_float.min(axis=(0, 1), keepdims=True)
band_max = cube_float.max(axis=(0, 1), keepdims=True)
band_range = band_max - band_min

constant_band_indices = np.flatnonzero(band_range.reshape(-1) == 0)
if len(constant_band_indices) > 0:
    raise ValueError(
        f"Constant spectral bands found: {constant_band_indices.tolist()}"
    )

normalized_cube = (cube_float - band_min) / band_range

assert normalized_cube.shape == cube.shape
assert normalized_cube.dtype == np.float32
assert not np.shares_memory(normalized_cube, cube)
assert np.allclose(normalized_cube.min(axis=(0, 1)), 0.0)
assert np.allclose(normalized_cube.max(axis=(0, 1)), 1.0)

print("Normalized cube shape:", normalized_cube.shape)
print("Normalized cube dtype:", normalized_cube.dtype)
print("Normalized range:", normalized_cube.min(), normalized_cube.max())
print("Raw cube remains unchanged:", cube.dtype, cube.min(), cube.max())

### Reflect Padding

In [ ]:
original_cube_shape = cube.shape
original_cube_dtype = cube.dtype

padded_cube = np.pad(
    normalized_cube,
    (
        (PATCH_RADIUS, PATCH_RADIUS),
        (PATCH_RADIUS, PATCH_RADIUS),
        (0, 0),
    ),
    mode="reflect",
)

expected_padded_shape = (
    normalized_cube.shape[0] + 2 * PATCH_RADIUS,
    normalized_cube.shape[1] + 2 * PATCH_RADIUS,
    normalized_cube.shape[2],
)
restored_normalized_cube = padded_cube[
    PATCH_RADIUS:-PATCH_RADIUS,
    PATCH_RADIUS:-PATCH_RADIUS,
    :,
]

assert cube.shape == original_cube_shape
assert cube.dtype == original_cube_dtype
assert padded_cube.shape == expected_padded_shape
assert padded_cube.dtype == normalized_cube.dtype
assert not np.shares_memory(normalized_cube, padded_cube)
assert np.array_equal(restored_normalized_cube, normalized_cube)

print("Normalized cube shape:", normalized_cube.shape)
print("Padded cube shape:", padded_cube.shape)
print("Raw cube remains unchanged.")

### Single Patch Extraction Check

In [ ]:
sample_row, sample_column, sample_label = loaded_split_tables[
    "train"
][0]

sample_row = int(sample_row)
sample_column = int(sample_column)
sample_label = int(sample_label)

sample_patch = padded_cube[
    sample_row:sample_row + int(PATCH_SIZE),
    sample_column:sample_column + int(PATCH_SIZE),
    :,
]

expected_patch_shape = (int(PATCH_SIZE), int(PATCH_SIZE), cube.shape[2])
center_spectrum = sample_patch[PATCH_RADIUS, PATCH_RADIUS, :]

assert sample_patch.shape == expected_patch_shape
assert np.array_equal(
    center_spectrum,
    normalized_cube[sample_row, sample_column, :],
)
assert ground_truth[sample_row, sample_column] == sample_label
assert sample_label != 0

print("Coordinate:", (sample_row, sample_column))
print("Target:", sample_label, class_names[sample_label])
print("Patch shape:", sample_patch.shape)
print("Center pixel and target verified.")

### Reusable Patch Extraction

In [29]:
def extract_patch(row, column):
    patch = padded_cube[
        row:row + int(PATCH_SIZE),
        column:column + int(PATCH_SIZE),
        :,
    ]

    expected_shape = (
        int(PATCH_SIZE),
        int(PATCH_SIZE),
        cube.shape[2],
    )
    if patch.shape != expected_shape:
        raise ValueError(
            f"Invalid patch shape {patch.shape} at {(row, column)}"
        )

    return patch


all_saved_samples = np.vstack(list(loaded_split_tables.values()))
saved_rows = all_saved_samples[:, 0]
saved_columns = all_saved_samples[:, 1]

border_mask = (
    (saved_rows < PATCH_RADIUS)
    | (saved_rows >= ground_truth.shape[0] - PATCH_RADIUS)
    | (saved_columns < PATCH_RADIUS)
    | (saved_columns >= ground_truth.shape[1] - PATCH_RADIUS)
)
border_row, border_column, border_label = all_saved_samples[border_mask][0]
border_row = int(border_row)
border_column = int(border_column)
border_label = int(border_label)

border_patch = extract_patch(border_row, border_column)
border_center = border_patch[PATCH_RADIUS, PATCH_RADIUS, :]

assert border_patch.shape == (
    int(PATCH_SIZE),
    int(PATCH_SIZE),
    cube.shape[2],
)
assert np.array_equal(
    border_center,
    normalized_cube[border_row, border_column, :],
)
assert ground_truth[border_row, border_column] == border_label

print("Border coordinate:", (border_row, border_column))
print("Target:", border_label, class_names[border_label])
print("Patch shape:", border_patch.shape)
print("Border patch verified.")

Border coordinate: (69, 2)
Target: 1 Asphalt
Patch shape: (15, 15, 103)
Border patch verified.


### PyTorch Patch Dataset

In [ ]:
import torch
from torch.utils.data import Dataset


class HyperspectralPatchDataset(Dataset):
    def __init__(self, padded_hsi_cube, split_table, patch_size):
        self.padded_hsi_cube = padded_hsi_cube
        self.coordinates = split_table[:, :2].astype(np.int64)
        self.targets = split_table[:, 2].astype(np.int64) - 1
        self.patch_size = int(patch_size)

        assert self.coordinates.shape == (len(split_table), 2)
        assert self.targets.min() >= 0
        assert self.targets.max() < 9

    def __len__(self):
        return len(self.coordinates)

    def __getitem__(self, index):
        row, column = self.coordinates[index]
        patch = self.padded_hsi_cube[
            row:row + self.patch_size,
            column:column + self.patch_size,
            :,
        ]
        patch = np.ascontiguousarray(
            patch.transpose(2, 0, 1),
            dtype=np.float32,
        )

        patch_tensor = torch.from_numpy(patch)
        target_tensor = torch.tensor(
            self.targets[index],
            dtype=torch.long,
        )
        return patch_tensor, target_tensor


train_dataset = HyperspectralPatchDataset(
    padded_cube, loaded_split_tables["train"], PATCH_SIZE
)
validation_dataset = HyperspectralPatchDataset(
    padded_cube, loaded_split_tables["validation"], PATCH_SIZE
)
test_dataset = HyperspectralPatchDataset(
    padded_cube, loaded_split_tables["test"], PATCH_SIZE
)

example_patch, example_target = train_dataset[0]
example_row, example_column = train_dataset.coordinates[0]

assert len(train_dataset) == 90
assert len(validation_dataset) == 45
assert len(test_dataset) == 42641
assert example_patch.shape == (cube.shape[2], 15, 15)
assert example_patch.dtype == torch.float32
assert example_target.dtype == torch.long
assert torch.equal(
    example_patch[:, PATCH_RADIUS, PATCH_RADIUS],
    torch.from_numpy(
        normalized_cube[example_row, example_column, :]
    ),
)

print("Dataset sizes:", len(train_dataset), len(validation_dataset), len(test_dataset))
print("Patch tensor shape:", tuple(example_patch.shape))
print("Target index:", example_target.item())
print("Original class label:", example_target.item() + 1)